# 03 — Retrieval Demo

Full pipeline: **PDF → extract → classify → chunk → embed → Qdrant + MongoDB → retrieve**

This notebook demonstrates the Week 3 storage & retrieval layer:
1. Ingest 2 PDFs (Java.pdf, C.pdf) into MongoDB + Qdrant
2. Query 5 ground-truth questions and inspect retrieved chunks

In [1]:
import logging
import os
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()  # loads .env from project root

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(name)s | %(message)s")

DATA_DIR = Path("../data")
print("PDFs available:", [f.name for f in DATA_DIR.glob("*.pdf")])
print("GOOGLE_API_KEY set:", bool(os.environ.get("GOOGLE_API_KEY")))
print("QDRANT_URL set:", bool(os.environ.get("QDRANT_URL")))
print("MONGO_URI set:", bool(os.environ.get("MONGO_URI", "default")))

PDFs available: ['C.pdf', 'Java.pdf', 'Python.pdf']
GOOGLE_API_KEY set: True
QDRANT_URL set: True
MONGO_URI set: True


## 1. Ingest PDFs

Run the full pipeline for each PDF: extract → classify → chunk → embed → store.

In [2]:
from hsc_edu.storage.ingest import ingest_pdf

n_java = ingest_pdf(DATA_DIR / "Java.pdf", subject="Lập trình Java", doc_id="java")
print(f"Ingested {n_java} chunks from Java.pdf")

INFO | hsc_edu.storage.ingest | Ingesting 'Java.pdf' (subject='Lập trình Java', doc_id='java')…
INFO | hsc_edu.core.extraction.pdf_detector | PDF detection: Java.pdf → text-based  (text_pages=231, scan_pages=10, ratio=0.96)
INFO | hsc_edu.core.extraction.text_extractor | Extracted 2121 blocks from 'Java.pdf' (2363 raw → 2121 after noise filter)
INFO | hsc_edu.core.classification.block_classifier | Loaded classification config from C:\Homework\Code File\Python Code File\de an\hsc_edu\config\subject_configs\default.yaml
INFO | hsc_edu.core.classification.block_classifier | Classified 2121 blocks: 210 headings, 1897 paragraphs, 14 special
INFO | hsc_edu.core.linking | Hierarchy links: 2377
INFO | hsc_edu.core.linking | Semantic graph built: 2377 total links
INFO | hsc_edu.core.chunking.text_chunker | Chunked 2121 blocks → 211 groups → 322 raw chunks → 322 final chunks
INFO | hsc_edu.core.assembly.context_assembler | Assembled context for 322 chunks
INFO | httpx | HTTP Request: GET https:/

Ingested 322 chunks from Java.pdf


In [3]:
n_c = ingest_pdf(DATA_DIR / "C.pdf", subject="Lập trình C", doc_id="c-lang")
print(f"Ingested {n_c} chunks from C.pdf")

INFO | hsc_edu.storage.ingest | Ingesting 'C.pdf' (subject='Lập trình C', doc_id='c-lang')…
INFO | hsc_edu.core.extraction.pdf_detector | PDF detection: C.pdf → text-based  (text_pages=97, scan_pages=5, ratio=0.95)
INFO | hsc_edu.core.extraction.text_extractor | Extracted 2328 blocks from 'C.pdf' (2430 raw → 2328 after noise filter)
INFO | hsc_edu.core.classification.block_classifier | Classified 2328 blocks: 94 headings, 2160 paragraphs, 74 special
INFO | hsc_edu.core.linking | Hierarchy links: 4604
INFO | hsc_edu.core.linking | Semantic graph built: 4604 total links
INFO | hsc_edu.core.chunking.text_chunker | Chunked 2328 blocks → 95 groups → 115 raw chunks → 115 final chunks
INFO | hsc_edu.core.assembly.context_assembler | Assembled context for 115 chunks
INFO | hsc_edu.storage.ingest | Embedding 115 chunks…
INFO | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"
INFO | httpx | HTTP Request: 

Ingested 115 chunks from C.pdf


## 2. Storage stats

In [4]:
from hsc_edu.storage.mongo_store import MongoChunkStore
from hsc_edu.storage.vector_store import QdrantVectorStore

mongo = MongoChunkStore()
qdrant = QdrantVectorStore()

print(f"MongoDB total chunks : {mongo.count()}")
print(f"MongoDB subjects     : {mongo.distinct_values('subject')}")
print(f"MongoDB doc_ids      : {mongo.distinct_values('doc_id')}")
print()
print(f"Qdrant collection    : {qdrant.collection_info()}")

INFO | httpx | HTTP Request: GET https://b789f557-a87e-425a-98ac-ce034dd4c6a0.us-east-1-1.aws.cloud.qdrant.io:6333/collections/hsc_edu_chunks/exists "HTTP/1.1 200 OK"
INFO | httpx | HTTP Request: GET https://b789f557-a87e-425a-98ac-ce034dd4c6a0.us-east-1-1.aws.cloud.qdrant.io:6333 "HTTP/1.1 200 OK"
INFO | httpx | HTTP Request: PUT https://b789f557-a87e-425a-98ac-ce034dd4c6a0.us-east-1-1.aws.cloud.qdrant.io:6333/collections/hsc_edu_chunks/index?wait=true&timeout=180 "HTTP/1.1 200 OK"
INFO | hsc_edu.storage.vector_store | Qdrant payload index ready: 'subject' (keyword)
INFO | httpx | HTTP Request: PUT https://b789f557-a87e-425a-98ac-ce034dd4c6a0.us-east-1-1.aws.cloud.qdrant.io:6333/collections/hsc_edu_chunks/index?wait=true&timeout=180 "HTTP/1.1 200 OK"
INFO | hsc_edu.storage.vector_store | Qdrant payload index ready: 'chapter' (keyword)
INFO | httpx | HTTP Request: PUT https://b789f557-a87e-425a-98ac-ce034dd4c6a0.us-east-1-1.aws.cloud.qdrant.io:6333/collections/hsc_edu_chunks/index?wait

MongoDB total chunks : 437
MongoDB subjects     : ['Lập trình C', 'Lập trình Java']
MongoDB doc_ids      : ['c-lang', 'java']



INFO | httpx | HTTP Request: GET https://b789f557-a87e-425a-98ac-ce034dd4c6a0.us-east-1-1.aws.cloud.qdrant.io:6333/collections/hsc_edu_chunks "HTTP/1.1 200 OK"


Qdrant collection    : {'name': 'hsc_edu_chunks', 'points_count': 437, 'status': 'green'}


## 3. Retrieval tests — ground truth questions

We test 5 questions from `data/note/ground_truth.md` to check if the
retrieved chunks contain the expected content.

In [5]:
from hsc_edu.storage.retrieval import retrieve_chunks

def show_results(query: str, results: list, max_preview: int = 200):
    print(f"\n{'=' * 80}")
    print(f"QUERY: {query}")
    print(f"{'=' * 80}")
    if not results:
        print("  (no results)")
        return
    for i, (chunk, score) in enumerate(results):
        path_str = " > ".join(chunk.section_path) if chunk.section_path else "(no heading)"
        preview = chunk.text[:max_preview].replace("\n", " ↵ ")
        if len(chunk.text) > max_preview:
            preview += " …"
        print(f"\n  [{i+1}] score={score:.4f}  │  {chunk.subject}  │  pages {chunk.page_start}–{chunk.page_end}")
        print(f"      Section: {path_str}")
        print(f"      Text   : {preview}")

In [6]:
# Q1 (Java) — Bốn nguyên tắc OOP
q1 = "Bốn nguyên tắc trụ cột của lập trình hướng đối tượng là gì?"
r1 = retrieve_chunks(q1, subject="Lập trình Java", top_k=5)
show_results(q1, r1)

INFO | httpx | HTTP Request: GET https://b789f557-a87e-425a-98ac-ce034dd4c6a0.us-east-1-1.aws.cloud.qdrant.io:6333/collections/hsc_edu_chunks/exists "HTTP/1.1 200 OK"
INFO | httpx | HTTP Request: GET https://b789f557-a87e-425a-98ac-ce034dd4c6a0.us-east-1-1.aws.cloud.qdrant.io:6333 "HTTP/1.1 200 OK"
INFO | httpx | HTTP Request: PUT https://b789f557-a87e-425a-98ac-ce034dd4c6a0.us-east-1-1.aws.cloud.qdrant.io:6333/collections/hsc_edu_chunks/index?wait=true&timeout=180 "HTTP/1.1 200 OK"
INFO | hsc_edu.storage.vector_store | Qdrant payload index ready: 'subject' (keyword)
INFO | httpx | HTTP Request: PUT https://b789f557-a87e-425a-98ac-ce034dd4c6a0.us-east-1-1.aws.cloud.qdrant.io:6333/collections/hsc_edu_chunks/index?wait=true&timeout=180 "HTTP/1.1 200 OK"
INFO | hsc_edu.storage.vector_store | Qdrant payload index ready: 'chapter' (keyword)
INFO | httpx | HTTP Request: PUT https://b789f557-a87e-425a-98ac-ce034dd4c6a0.us-east-1-1.aws.cloud.qdrant.io:6333/collections/hsc_edu_chunks/index?wait


QUERY: Bốn nguyên tắc trụ cột của lập trình hướng đối tượng là gì?

  [1] score=0.7930  │  Lập trình Java  │  pages 14–14
      Section: 1.3. CÁC NGUYÊN TẮC TRỤ CỘT
      Text   : 1.3. CÁC NGUYÊN TẮC TRỤ CỘT ↵  ↵ Lập trình hướng đối tượng có ba nguyên tắc trụ cột: đóng gói, thừa kế và đa  ↵ hình,  còn trừu tượng hóa là khái niệm nền tảng. ↵  ↵ Trừu tượng hóa (abstraction) là một cơ chế c …

  [2] score=0.7664  │  Lập trình Java  │  pages 14–16
      Section: 1.3. CÁC NGUYÊN TẮC TRỤ CỘT
      Text   : 1.3. CÁC NGUYÊN TẮC TRỤ CỘT ↵  ↵ Nói theo phương diện lập trình, nhìn từ bên ngoài một mô-đun chỉ thấy được  ↵ các giao diện. Các lập trình viên tự do cài đặt chi tiết bên trong, với ràng buộc duy  ↵ nhất là  …

  [3] score=0.7651  │  Lập trình Java  │  pages 14–15
      Section: 1.3. CÁC NGUYÊN TẮC TRỤ CỘT
      Text   : 1.3. CÁC NGUYÊN TẮC TRỤ CỘT ↵  ↵ Phương pháp hướng đối tượng trừu tượng hóa thế giới thực thành các đối  ↵ tượng và tương tác giữa chúng với các đối tượng khác. Việc mô 

In [7]:
# Q2 (Java) — Tính đa hình
q2 = "Tính đa hình trong lập trình hướng đối tượng được hiểu như thế nào?"
r2 = retrieve_chunks(q2, subject="Lập trình Java", top_k=5)
show_results(q2, r2)

INFO | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"
INFO | hsc_edu.storage.embedding | Embedded 1 texts → 1 vectors (dim=768, model=gemini-embedding-001)
INFO | httpx | HTTP Request: POST https://b789f557-a87e-425a-98ac-ce034dd4c6a0.us-east-1-1.aws.cloud.qdrant.io:6333/collections/hsc_edu_chunks/points/query "HTTP/1.1 200 OK"
INFO | hsc_edu.storage.retrieval | Retrieved 5 chunks for query='Tính đa hình trong lập trình hướng đối tượng được hiểu như thế nào?' (top score=0.7950)



QUERY: Tính đa hình trong lập trình hướng đối tượng được hiểu như thế nào?

  [1] score=0.7950  │  Lập trình Java  │  pages 110–113
      Section: 7.7. ĐA HÌNH
      Text   : 7.7. ĐA HÌNH ↵  ↵ Tóm lại, đa hình là gì? Theo nghĩa tổng quát, đa hình là khả năng tồn tại ở nhiều  ↵ hình thức. Trong hướng đối tượng, đa hình đi kèm với quan hệ thừa kế và có hai đặc  ↵ điểm sau: (1) các đ …

  [2] score=0.7899  │  Lập trình Java  │  pages 14–17
      Section: 1.3. CÁC NGUYÊN TẮC TRỤ CỘT
      Text   : 1.3. CÁC NGUYÊN TẮC TRỤ CỘT ↵  ↵ chẳng hạn Shape, ta có thể dùng quan hệ thừa kế để xây dựng các lớp mô hình hóa  ↵ các khái niệm cụ thể hơn, chẳng hạn Circle, Triangle. Bằng cách này, ta có thể sử  ↵ dụng gi …

  [3] score=0.7711  │  Lập trình Java  │  pages 110–112
      Section: 7.7. ĐA HÌNH
      Text   : 7.7. ĐA HÌNH ↵  ↵ class Vet { ↵   public void giveShot(Animal a) { ↵     // give a a shot, vaccination for example ↵     a.makeNoise(); ↵   } ↵ } ↵  ↵ Vet v = new Vet(); ↵ Dog d = new Dog()

In [8]:
# Q3 (C) — Thuật giải là gì
q3 = "Giải thuật (Algorithm) là gì và nó dựa trên những cấu trúc điều khiển cơ bản nào?"
r3 = retrieve_chunks(q3, subject="Lập trình C", top_k=5)
show_results(q3, r3)

INFO | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"
INFO | hsc_edu.storage.embedding | Embedded 1 texts → 1 vectors (dim=768, model=gemini-embedding-001)
INFO | httpx | HTTP Request: POST https://b789f557-a87e-425a-98ac-ce034dd4c6a0.us-east-1-1.aws.cloud.qdrant.io:6333/collections/hsc_edu_chunks/points/query "HTTP/1.1 200 OK"
INFO | hsc_edu.storage.retrieval | Retrieved 5 chunks for query='Giải thuật (Algorithm) là gì và nó dựa trên những cấu trúc điều khiển cơ bản nào' (top score=0.7228)



QUERY: Giải thuật (Algorithm) là gì và nó dựa trên những cấu trúc điều khiển cơ bản nào?

  [1] score=0.7228  │  Lập trình C  │  pages 6–7
      Section: (no heading)
      Text   : hạn các bước thì đạt được mục tiêu. ↵  ↵ 1.1.2. Chương trình (Program) ↵  ↵ Là một tập hợp các mô tả, các phát biểu, nằm trong một hệ thống qui ước nhằm điều khiển ↵  ↵ máy tính làm việc. Chương trình có thể hiểu …

  [2] score=0.6719  │  Lập trình C  │  pages 48–48
      Section: Chương 6 : CẤU TRÚC VÒNG LẶP
      Text   : Chương 6 : CẤU TRÚC VÒNG LẶP

  [3] score=0.6541  │  Lập trình C  │  pages 0–3
      Section: (no heading)
      Text   : BÀI GI ↵  ↵ Ả ↵  ↵ NG ↵  ↵ L ↵  ↵ Ậ ↵  ↵ P TRÌNH ↵  ↵ C ↵  ↵ MỤC LỤC ↵  ↵ MỤC LỤC .............................................................................................................................  ↵ 2 ↵  ↵ Chương 1: NGÔN NGỮ L …

  [4] score=0.6510  │  Lập trình C  │  pages 2–4
      Section: (no heading)
      Text   : 5.1. Lệnh if ......................

In [9]:
# Q4 (C) — Kiểu dữ liệu cơ bản
q4 = "Liệt kê các kiểu dữ liệu cơ bản trong C và kích thước bộ nhớ tương ứng?"
r4 = retrieve_chunks(q4, subject="Lập trình C", top_k=5)
show_results(q4, r4)

INFO | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"
INFO | hsc_edu.storage.embedding | Embedded 1 texts → 1 vectors (dim=768, model=gemini-embedding-001)
INFO | httpx | HTTP Request: POST https://b789f557-a87e-425a-98ac-ce034dd4c6a0.us-east-1-1.aws.cloud.qdrant.io:6333/collections/hsc_edu_chunks/points/query "HTTP/1.1 200 OK"
INFO | hsc_edu.storage.retrieval | Retrieved 5 chunks for query='Liệt kê các kiểu dữ liệu cơ bản trong C và kích thước bộ nhớ tương ứng?' (top score=0.7559)



QUERY: Liệt kê các kiểu dữ liệu cơ bản trong C và kích thước bộ nhớ tương ứng?

  [1] score=0.7559  │  Lập trình C  │  pages 25–25
      Section: 3.3. Kiểu dữ liệu
      Text   : 3.3. Kiểu dữ liệu ↵  ↵ Có 4 kiểu dữ liệu cơ bản trong C là: char, int, float, double. Tuy nhiên từ 4 kiểu dữ liệu cơ ↵  ↵ bản, C có cung cấp một số kiểu mở rộng khác chi tiết trong bảng.

  [2] score=0.7019  │  Lập trình C  │  pages 4–6
      Section: (no heading)
      Text   : 8.4.3. Khởi tạo chuỗi .................................................................................................... 68 ↵  ↵ 8.4.4. Mảng chuỗi ....................................................... …

  [3] score=0.6915  │  Lập trình C  │  pages 62–64
      Section: Chương 9. CON TRỎ TRONG C > 9.1. Giới thiệu
      Text   : 9.1. Giới thiệu ↵  ↵ Các biến chúng ta đã biết và sử dụng trước đây đều là biến có kích thước và kiểu dữ liệu xác ↵  ↵ định. Người ta gọi các biến kiểu này là biến tĩnh. Khi khai báo biến tĩnh, một lượng ô n

In [10]:
# Q5 (cross-subject, no filter) — Thừa kế
q5 = "Thế nào là quan hệ thừa kế giữa lớp cha và lớp con?"
r5 = retrieve_chunks(q5, top_k=5)
show_results(q5, r5)

INFO | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"
INFO | hsc_edu.storage.embedding | Embedded 1 texts → 1 vectors (dim=768, model=gemini-embedding-001)
INFO | httpx | HTTP Request: POST https://b789f557-a87e-425a-98ac-ce034dd4c6a0.us-east-1-1.aws.cloud.qdrant.io:6333/collections/hsc_edu_chunks/points/query "HTTP/1.1 200 OK"
INFO | hsc_edu.storage.retrieval | Retrieved 5 chunks for query='Thế nào là quan hệ thừa kế giữa lớp cha và lớp con?' (top score=0.7902)



QUERY: Thế nào là quan hệ thừa kế giữa lớp cha và lớp con?

  [1] score=0.7902  │  Lập trình Java  │  pages 102–103
      Section: 7.1. QUAN HỆ THỪA KẾ
      Text   : 7.1. QUAN HỆ THỪA KẾ ↵  ↵ Nhớ lại ví dụ đầu tiên về lập trình hướng đối tượng tại Ch-¬ng 1. Trong đó, Dậu  ↵ xây dựng 4 lớp: Square (hình vuông), Circle (đường tròn), Triangle (hình tam giác),  ↵ và Amoeba (h …

  [2] score=0.7798  │  Lập trình Java  │  pages 102–103
      Section: 7.1. QUAN HỆ THỪA KẾ
      Text   : 7.1. QUAN HỆ THỪA KẾ ↵  ↵ Khi ta dùng quan hệ thừa kế trong thiết kế, ta đặt các phần mã dùng chung tại  ↵ một lớp và coi đó là lớp cha – lớp dùng chung trừu tượng hơn, các lớp cụ thể hơn là  ↵ các lớp con. C …

  [3] score=0.7597  │  Lập trình Java  │  pages 109–109
      Section: 7.5. KHI NÀO NÊN DÙNG QUAN HỆ THỪA KẾ?
      Text   : 7.5. KHI NÀO NÊN DÙNG QUAN HỆ THỪA KẾ? ↵  ↵ Mục này liệt kê một số quy tắc hướng dẫn việc sử dụng quan hệ thừa kế trong  ↵ thiết kế. Tại thời điểm này, ta tạm bằng lòng với việ

## 4. Summary

| # | Question | Subject filter | Expected topic | Relevant? |
|---|----------|---------------|----------------|------------|
| 1 | 4 nguyên tắc OOP | Java | OOP principles | |
| 2 | Đa hình | Java | Polymorphism | |
| 3 | Thuật giải | C | Algorithm definition | |
| 4 | Kiểu dữ liệu | C | Data types | |
| 5 | Thừa kế | (none) | Inheritance | |

Fill in the **Relevant?** column after inspecting the results above.